In [ ]:
import os
import pandas as pd

folder = r"C:\Users\DELL\Documents\project_data\data"
file_path = os.path.join(folder, "dirty_cafe_sales.csv")

df = pd.read_csv(file_path)
# -----------------------------
# 1. Fix column names
# -----------------------------
df.columns = df.columns.str.strip()
print(df.head())
# -----------------------------
# 2. Replace bad values
# -----------------------------
df.replace("ERROR", pd.NA, inplace=True)
df.replace("UNKNOWN", pd.NA, inplace=True)

# -----------------------------
# 3. Convert numeric columns
# -----------------------------
num_cols = ["Quantity", "Price Per Unit", "Total Spent"]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# -----------------------------
# 4. Fix Total Spent if missing
# -----------------------------
df["Total Spent"] = df["Total Spent"].fillna(
    df["Quantity"] * df["Price Per Unit"]
)

# -----------------------------
# 5. Fill numeric nulls with median
# -----------------------------
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# -----------------------------
# 6. Fix categorical columns
# -----------------------------
cat_cols = ["Item", "Payment Method", "Location"]

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# -----------------------------
# 7. Convert date column
# -----------------------------
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

# -----------------------------
# 8. Create extra BI columns
# -----------------------------
df["Year"] = df["Transaction Date"].dt.year
df["Month"] = df["Transaction Date"].dt.month
df["Day"] = df["Transaction Date"].dt.day
df["Weekday"] = df["Transaction Date"].dt.day_name()

# Fill missing dates with the most frequent date (mode)
mode_date = df["Transaction Date"].mode()[0]
df["Transaction Date"] = df["Transaction Date"].fillna(mode_date)

# Recreate time columns correctly as integers
df["Year"] = df["Transaction Date"].dt.year.astype(int)
df["Month"] = df["Transaction Date"].dt.month.astype(int)
df["Day"] = df["Transaction Date"].dt.day.astype(int)
df["Weekday"] = df["Transaction Date"].dt.day_name()

print("Clean dataset ready for Power BI ✅")

  Transaction ID    Item Quantity Price Per Unit Total Spent  Payment Method  \
0    TXN_1961373  Coffee        2            2.0         4.0     Credit Card   
1    TXN_4977031    Cake        4            3.0        12.0            Cash   
2    TXN_4271903  Cookie        4            1.0       ERROR     Credit Card   
3    TXN_7034554   Salad        2            5.0        10.0         UNKNOWN   
4    TXN_3160411  Coffee        2            2.0         4.0  Digital Wallet   

   Location Transaction Date  
0  Takeaway       2023-09-08  
1  In-store       2023-05-16  
2  In-store       2023-07-19  
3   UNKNOWN       2023-04-27  
4  In-store       2023-06-11  
Clean dataset ready for Power BI ✅


In [12]:
# -----------------------------
# 9. DATA QUALITY CHECK (before save)
# -----------------------------
print("\n🔍 DATA QUALITY REPORT")

print("Rows:", len(df))
print("\nMissing values per column:")
print(df.isna().sum())

print("\nData types:")
print(df.dtypes)

# Check for negative or zero values
print("\nInvalid numeric values:")
print("Quantity <= 0 :", (df["Quantity"] <= 0).sum())
print("Price Per Unit <= 0 :", (df["Price Per Unit"] <= 0).sum())
print("Total Spent <= 0 :", (df["Total Spent"] <= 0).sum())

# Check date problems
print("\nMissing Transaction Dates:", df["Transaction Date"].isna().sum())

# Check duplicates
print("\nDuplicate Transaction IDs:", df["Transaction ID"].duplicated().sum())

print("\n✅ If all above look OK, file is safe for Power BI.\n")

# -----------------------------
# 10. Save clean file
# -----------------------------
df.to_csv(os.path.join(folder, "transactions_powerbi_ready.csv"), index=False)

print("File saved successfully ✅")


🔍 DATA QUALITY REPORT
Rows: 10000

Missing values per column:
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Year                0
Month               0
Day                 0
Weekday             0
dtype: int64

Data types:
Transaction ID              object
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
Year                         int64
Month                        int64
Day                          int64
Weekday                     object
dtype: object

Invalid numeric values:
Quantity <= 0 : 0
Price Per Unit <= 0 : 0
Total Spent <= 0 : 0

Missing Transaction Dates: 0

Duplicate Transaction IDs: 0

✅ If all above look OK, file is safe for Power BI.

File saved s